# NFL Forecasting Lab v2
## Position-specific models • walk-forward backtests • next game • rest of season • next year

This notebook consumes the NFL public-data warehouse and coaching tables. It builds a repeatable modeling workflow, creates position-specific models when there is enough data, compares baselines with machine-learning models, uses chronological evaluation, estimates prediction intervals, and exports model artifacts and forecast templates.

**Important limits**
- This notebook cannot make unavailable sources appear. Audit coverage before modeling.
- Forecasts are statistical estimates, not guarantees or betting advice.
- Results are only valid if target columns, player-game grain, pregame feature timing, and schedule joins are verified.
- Never use postgame information to predict that same game. Features derived from the target must be shifted to prior games.
- Models by position are only fit if that position has enough rows and distinct seasons.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json, warnings, re, math, os, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neural_network import MLPRegressor

DATA_ROOT = Path("nfl_data")           # match the collection notebook output directory
COACH_ROOT = DATA_ROOT / "coaching"
OUT = Path("nfl_forecasts")
MODEL_DIR = OUT / "models"
OUT.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
MIN_POSITION_ROWS = 250
MIN_TRAIN_SEASONS = 3
TARGET_OVERRIDE = None  # e.g. "receiving_yards"; None selects a suitable detected target
PREDICTION_INTERVAL_ALPHA = 0.20      # nominal 80% residual interval
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

def load_table(stem, roots=(DATA_ROOT, COACH_ROOT)):
    for root in roots:
        if root.exists():
            hits = sorted(list(root.rglob(f"{stem}.parquet")) + list(root.rglob(f"{stem}.csv")))
            if hits:
                p = hits[0]
                try:
                    return pd.read_parquet(p) if p.suffix == ".parquet" else pd.read_csv(p, low_memory=False)
                except Exception as e:
                    print(f"Could not load {p}: {e}")
    return pd.DataFrame()

def first_col(df, candidates):
    return next((c for c in candidates if c in df.columns), None)

def safe_num(s):
    return pd.to_numeric(s, errors="coerce")

## 1. Inventory and data quality gates

The collector exports multiple tables independently. We load them without assuming every source exists, and write a table inventory. Confirm row counts, date ranges, and key uniqueness before training.

In [2]:
TABLES = [
    "pbp","schedules","player_stats","player_stats_weekly","team_stats","players","rosters",
    "weekly_rosters","snap_counts","ngs_passing","ngs_receiving","ngs_rushing","participation",
    "injuries","depth_charts","draft_picks","combine","pfr_passing","pfr_rushing","pfr_receiving",
    "team_game_tendencies","team_season_tendencies","coaches","coach_tenures","coaching_changes"
]
D = {name: load_table(name) for name in TABLES}
inventory = pd.DataFrame([{
    "table": k, "rows": len(v), "columns": len(v.columns), "loaded": not v.empty,
    "column_names": ", ".join(map(str, v.columns[:25]))
} for k,v in D.items()])
display(inventory)
inventory.to_csv(OUT / "input_inventory.csv", index=False)

,table,rows,columns,loaded,column_names
0,pbp,1293659,372,True,"play_id, game_id, old_game_id, home_team, away..."
1,schedules,7548,46,True,"game_id, season, game_type, week, gameday, wee..."
2,player_stats,481734,150,True,"player_id, player_name, player_display_name, p..."
3,player_stats_weekly,0,0,False,
4,team_stats,14691,138,True,"season, week, team, season_type, game_id, oppo..."
5,players,24854,39,True,"gsis_id, display_name, common_first_name, firs..."
6,rosters,142728,36,True,"season, team, position, depth_chart_position, ..."
7,weekly_rosters,921705,36,True,"season, team, position, depth_chart_position, ..."
8,snap_counts,332075,16,True,"game_id, pfr_game_id, season, game_type, week,..."
9,ngs_passing,0,0,False,


## 2. Create a canonical player-game table

The canonical grain must be one row per player per game. This cell supports common nflverse field names but cannot infer every custom schema. It combines weekly/player-stat sources only when available, then deduplicates by ID/game keys. Inspect `player_games.columns` and adjust the mappings if your exports differ.

In [3]:
def normalize_player_table(df, source):
    if df.empty: return pd.DataFrame()
    x = df.copy()
    aliases = {
        "player_id": ["player_id","gsis_id","nfl_id","pfr_id"],
        "player_name": ["player_display_name","player_name","display_name","name"],
        "team": ["recent_team","team","posteam","club_code"],
        "position": ["position","position_group","pos"],
        "season": ["season","season_year"],
        "week": ["week","game_week"],
        "game_id": ["game_id","gsis_game_id"],
        "opponent": ["opponent_team","opponent","defteam"],
        "gameday": ["gameday","game_date","date"],
    }
    for dest, options in aliases.items():
        col = first_col(x, options)
        if col and dest not in x.columns: x[dest] = x[col]
    x["source_table"] = source
    for c in ["season","week"]:
        if c in x: x[c] = safe_num(x[c])
    if "player_id" not in x and "player_name" not in x: return pd.DataFrame()
    return x

parts=[]
for name in ["player_stats_weekly","player_stats"]:
    part=normalize_player_table(D.get(name,pd.DataFrame()),name)
    if not part.empty: parts.append(part)

player_games = pd.concat(parts, ignore_index=True, sort=False) if parts else pd.DataFrame()
if not player_games.empty:
    keys=[c for c in ["player_id","season","week","game_id"] if c in player_games.columns]
    if len(keys)>=3:
        player_games=player_games.sort_values(keys).drop_duplicates(keys,keep="first")
    if "position" not in player_games:
        roster=D.get("weekly_rosters",pd.DataFrame())
        if roster.empty: roster=D.get("rosters",pd.DataFrame())
        rid=first_col(roster,["player_id","gsis_id","nfl_id"])
        rpos=first_col(roster,["position","position_group","pos"])
        if rid and rpos and "player_id" in player_games:
            roster_map=roster[[rid,rpos]].dropna().drop_duplicates(rid).rename(columns={rid:"player_id",rpos:"position"})
            player_games=player_games.merge(roster_map,on="player_id",how="left")
print("player_games shape:",player_games.shape)
display(player_games.head())

player_games shape: (481734, 152)


,player_id,player_name,player_display_name,position,position_group,headshot_url,season,week,season_type,game_id,team,opponent_team,completions,attempts,passing_yards,passing_tds,passing_interceptions,sacks_suffered,sack_yards_lost,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_cpoe,passing_2pt_conversions,pacr,passing_10,passing_16,passing_20,passing_40,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,rushing_10,rushing_12,rushing_20,rushing_40,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,receiving_10,receiving_16,receiving_20,receiving_40,racr,...,fumbles_forced_by_opp,fumbles_not_forced,fumbles_out_of_bounds,fumbles_total,fumbles_lost_total,punt_returns,punt_return_yards,kickoff_returns,kickoff_return_yards,fg_made,fg_att,fg_missed,fg_blocked,fg_long,fg_pct,fg_made_0_19,fg_made_20_29,fg_made_30_39,fg_made_40_49,fg_made_50_59,fg_made_60_,fg_missed_0_19,fg_missed_20_29,fg_missed_30_39,fg_missed_40_49,fg_missed_50_59,fg_missed_60_,fg_made_list,fg_missed_list,fg_blocked_list,fg_made_distance,fg_missed_distance,fg_blocked_distance,pat_made,pat_att,pat_missed,pat_blocked,pat_pct,gwfg_made,gwfg_att,gwfg_missed,gwfg_blocked,gwfg_distance,pt_att,pt_blocked,pt_long,pt_yards,pt_inside_20,pt_out_of_bounds,pt_downed,pt_touchback,pt_fair_caught,pt_returned,pt_return_yards,pt_return_tds,pt_net_yards,fantasy_points,fantasy_points_ppr,opponent,source_table
0,0,None,None,None,None,None,1999,1,REG,1999_01_MIN_ATL,ATL,MIN,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,NaN,...,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0.0,0.0,MIN,player_stats
925,0,None,None,None,None,None,1999,2,REG,1999_02_PIT_BAL,BAL,PIT,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,NaN,...,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0.0,0.0,PIT,player_stats
1899,0,None,None,None,None,None,1999,3,REG,1999_03_CLE_BAL,CLE,BAL,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,NaN,...,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0.0,0.0,BAL,player_stats
2809,0,None,None,None,None,None,1999,4,REG,1999_04_BAL_ATL,ATL,BAL,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,NaN,...,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0.0,0.0,BAL,player_stats
3709,0,None,None,None,None,None,1999,5,REG,1999_05_NYG_ARI,NYG,ARI,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,NaN,...,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0.0,0.0,ARI,player_stats


## 3. Choose the outcome and audit player-game uniqueness

Default target selection is convenience only. Pick a meaningful numeric target for the question you want to answer. For example, fit separate models for passing yards, rushing yards, receiving yards, receptions, touchdowns, and fantasy points rather than combining incomparable outcomes.

In [4]:
TARGET_OPTIONS=["fantasy_points_ppr","fantasy_points","passing_yards","rushing_yards","receiving_yards",
                "receptions","targets","carries","passing_tds","rushing_tds","receiving_tds"]
TARGET=TARGET_OVERRIDE or (next((c for c in TARGET_OPTIONS if c in player_games.columns),None) if not player_games.empty else None)
print("Detected target:",TARGET)
if not player_games.empty:
    print("Rows:",len(player_games))
    if all(c in player_games for c in ["player_id","season","week"]):
        print("Duplicate player-season-week rows:",int(player_games.duplicated(["player_id","season","week"]).sum()))
    print("Position counts:")
    display(player_games["position"].value_counts(dropna=False).to_frame("rows") if "position" in player_games else pd.DataFrame())
if TARGET:
    player_games[TARGET]=safe_num(player_games[TARGET])
    display(player_games[[c for c in ["player_name","position","team","season","week",TARGET] if c in player_games]].head())

Detected target: fantasy_points_ppr
Rows: 481734
Duplicate player-season-week rows: 0
Position counts:


,rows
position,
WR,62526
DE,42106
RB,41305
LB,39509
CB,34832
DT,32655
DB,32547
TE,30359
OLB,23615


,player_name,position,team,season,week,fantasy_points_ppr
0,None,None,ATL,1999,1,0.0
925,None,None,BAL,1999,2,0.0
1899,None,None,CLE,1999,3,0.0
2809,None,None,ATL,1999,4,0.0
3709,None,None,NYG,1999,5,0.0


## 4. Leakage-safe feature engineering

For each player, lagged rolling averages and variability are calculated from prior games only. Current-game targets are never included in their own features. The initial feature set is intentionally conservative; additional pregame features should be joined using timestamp-aware rules.

In [5]:
def build_features(df, target, windows=(3,5,8)):
    if df.empty or target not in df: return pd.DataFrame()
    x=df.copy()
    idcol="player_id" if "player_id" in x else "player_name"
    if idcol not in x: raise ValueError("Player ID or name required")
    order=[c for c in [idcol,"season","week","gameday","game_id"] if c in x]
    x=x.sort_values(order).reset_index(drop=True)
    grp=x.groupby(idcol,dropna=False)[target]
    lag=grp.shift(1)
    for w in windows:
        x[f"prior_mean_{w}"]=lag.groupby(x[idcol],dropna=False).transform(lambda s:s.rolling(w,min_periods=1).mean())
        x[f"prior_std_{w}"]=lag.groupby(x[idcol],dropna=False).transform(lambda s:s.rolling(w,min_periods=2).std())
    x["prior_mean_career"]=grp.transform(lambda s:s.shift(1).expanding(min_periods=1).mean())
    x["prior_games"]=grp.transform(lambda s:s.shift(1).expanding().count())
    # Season/week are retained as calendar context, not future outcomes.
    return x

model_data=build_features(player_games,TARGET) if TARGET and not player_games.empty else pd.DataFrame()
ROLL_FEATURES=[c for c in model_data.columns if c.startswith("prior_")]
print("Feature columns:",ROLL_FEATURES)
display(model_data[[c for c in ["player_name","position","season","week",TARGET] if c in model_data]+ROLL_FEATURES].head(12) if not model_data.empty else pd.DataFrame())

Feature columns: ['prior_mean_3', 'prior_std_3', 'prior_mean_5', 'prior_std_5', 'prior_mean_8', 'prior_std_8', 'prior_mean_career', 'prior_games']


,player_name,position,season,week,fantasy_points_ppr,prior_mean_3,prior_std_3,prior_mean_5,prior_std_5,prior_mean_8,prior_std_8,prior_mean_career,prior_games
0,None,None,1999,1,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
1,None,None,1999,2,0.0,0.0,NaN,0.0,NaN,0.0,NaN,0.0,1.0
2,None,None,1999,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0
3,None,None,1999,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0
4,None,None,1999,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0
5,None,None,1999,6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0
6,None,None,1999,7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0
7,None,None,1999,8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0
8,None,None,1999,9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0
9,None,None,1999,10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.0


## 5. Optional contextual features

These helpers add team tendency features and selected stable player metadata where safe. Join keys and time validity must be checked. Do not join a full-season aggregate to earlier games in that season; it leaks future information. The season tendency table is therefore reserved for next-season summaries unless you replace it with lagged game-level aggregates.

In [6]:
def add_team_game_tendencies(frame):
    tend=D.get("team_game_tendencies",pd.DataFrame())
    if frame.empty or tend.empty: return frame
    team=first_col(tend,["team","posteam"])
    season=first_col(tend,["season","_season"])
    week=first_col(tend,["week","_week"])
    if not team or not season or not week or not all(c in frame for c in ["team","season","week"]):
        print("Skipping team tendency join: compatible team/season/week keys not found")
        return frame
    value_cols=[c for c in ["pass_rate","rush_rate","mean_epa","success_rate"] if c in tend.columns]
    if not value_cols:
        print("Skipping team tendency join: no expected tendency columns found")
        return frame
    t=tend[[team,season,week]+value_cols].copy().rename(columns={team:"team",season:"season",week:"week"})
    # A same-game team tendency would reveal play calling after kickoff. Shift each team's
    # game-level metrics by one game before joining them to player-game rows.
    t=t.drop_duplicates(["team","season","week"]).sort_values(["team","season","week"])
    for c in value_cols:
        t[f"prior_team_{c}"]=t.groupby("team",dropna=False)[c].shift(1)
    lag_cols=[f"prior_team_{c}" for c in value_cols]
    t=t[["team","season","week"]+lag_cols]
    return frame.merge(t,on=["team","season","week"],how="left",validate="many_to_one")

model_data=add_team_game_tendencies(model_data) if not model_data.empty else model_data
ROLL_FEATURES=[c for c in model_data.columns if c.startswith("prior_")]
SAFE_CONTEXT=[c for c in model_data.columns if c.startswith("prior_team_")]
FEATURE_CANDIDATES=ROLL_FEATURES+SAFE_CONTEXT

## 6. Position-specific walk-forward evaluation

Each position is evaluated by holding out the latest season and training only on prior seasons. We compare a training-mean baseline, Ridge regression, Extra Trees, HistGradientBoosting, and an MLP neural network. All models share the same chronological test rows. This first pass uses one target at a time and the available lag/context features.

In [7]:
def make_preprocessor(df, features):
    numeric=[c for c in features if pd.api.types.is_numeric_dtype(df[c])]
    categorical=[c for c in features if c not in numeric]
    transformers=[]
    if numeric:
        transformers.append(("num",Pipeline([("imputer",SimpleImputer(strategy="median")),("scale",StandardScaler())]),numeric))
    if categorical:
        transformers.append(("cat",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),
                                             ("onehot",OneHotEncoder(handle_unknown="ignore"))]),categorical))
    return ColumnTransformer(transformers,remainder="drop")

def make_models():
    return {
        "mean_baseline": DummyRegressor(strategy="mean"),
        "ridge": Ridge(alpha=10.0),
        "extra_trees": ExtraTreesRegressor(n_estimators=300,min_samples_leaf=3,max_features=1.0,random_state=SEED,n_jobs=-1),
        "hist_gradient_boosting": HistGradientBoostingRegressor(max_iter=250,learning_rate=0.05,l2_regularization=1.0,random_state=SEED),
        "mlp_neural_network": MLPRegressor(hidden_layer_sizes=(64,32),activation="relu",alpha=1.0,learning_rate_init=0.001,
                                           early_stopping=True,max_iter=500,random_state=SEED)
    }

def split_latest_season(df):
    seasons=sorted(pd.to_numeric(df["season"],errors="coerce").dropna().unique())
    if len(seasons)<3: return None
    test_season=seasons[-1]
    train=df[pd.to_numeric(df["season"],errors="coerce")<test_season].copy()
    test=df[pd.to_numeric(df["season"],errors="coerce")==test_season].copy()
    if train.empty or test.empty: return None
    return train,test,test_season

def metric_row(y,pred,model,position,season):
    return {"position":position,"model":model,"test_season":season,"n_test":len(y),
            "MAE":float(mean_absolute_error(y,pred)),"RMSE":float(np.sqrt(mean_squared_error(y,pred))),
            "R2":float(r2_score(y,pred)) if len(y)>1 else np.nan}

results=[]
fitted={}
prediction_frames=[]
if not model_data.empty and TARGET and "season" in model_data and "position" in model_data:
    base_features=[c for c in FEATURE_CANDIDATES if c in model_data and c != TARGET]
    # Exclude IDs, labels, future/current outcomes, and raw season/week from feature matrix.
    base_features=[c for c in base_features if c not in {"player_id","player_name","team","position","season","week","game_id","gameday"}]
    for pos, group in model_data.groupby("position",dropna=True):
        group=group[pd.to_numeric(group[TARGET],errors="coerce").notna()].copy()
        group=group[group[base_features].notna().any(axis=1)] if base_features else group.iloc[0:0]
        if len(group)<MIN_POSITION_ROWS: 
            print(f"Skip {pos}: only {len(group)} rows (minimum {MIN_POSITION_ROWS})")
            continue
        split=split_latest_season(group)
        if split is None:
            print(f"Skip {pos}: need at least 3 seasons")
            continue
        train,test,test_season=split
        # Drop rows with no history features; imputer handles missing individual features.
        if not base_features: continue
        Xtr,ytr=train[base_features],train[TARGET].astype(float)
        Xte,yte=test[base_features],test[TARGET].astype(float)
        for name, estimator in make_models().items():
            pipe=Pipeline([("prep",make_preprocessor(train,base_features)),("model",estimator)])
            try:
                pipe.fit(Xtr,ytr)
                pred=pipe.predict(Xte)
                results.append(metric_row(yte,pred,name,str(pos),test_season))
                if name!="mean_baseline":
                    fitted[(str(pos),name)]={"pipeline":pipe,"features":base_features,"target":TARGET,
                                             "train_seasons":sorted(train.season.dropna().unique().tolist()),
                                             "test_season":int(test_season)}
                if name=="extra_trees":
                    p=test[[c for c in ["player_id","player_name","position","team","season","week","game_id",TARGET] if c in test]].copy()
                    p["prediction"]=pred
                    p["residual"]=p[TARGET]-p["prediction"]
                    p["model"]=name
                    prediction_frames.append(p)
            except Exception as e:
                print(f"Model {name} failed for {pos}: {e}")
metrics=pd.DataFrame(results)
display(metrics.sort_values(["position","MAE"]) if not metrics.empty else metrics)
metrics.to_csv(OUT/"position_model_metrics.csv",index=False)

C:\Users\Stephen\anaconda3\envs\gpu\lib\site-packages\joblib\externals\loky\backend\context.py:110: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\Stephen\anaconda3\envs\gpu\lib\site-packages\joblib\externals\loky\backend\context.py", line 199, in _count_physical_cores
    cpu_info = subprocess.run(
  File "C:\Users\Stephen\anaconda3\envs\gpu\lib\subprocess.py", line 503, in run
    with Popen(*popenargs, **kwargs) as process:
  File "C:\Users\Stephen\anaconda3\envs\gpu\lib\subprocess.py", line 971, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\Stephen\anaconda3\envs\gpu\lib\subprocess.py", line 1456, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executabl

Skip OL: only 33 rows (minimum 250)


,position,model,test_season,n_test,MAE,RMSE,R2
3,C,hist_gradient_boosting,2026,61,0.022450,0.154081,-0.022460
2,C,extra_trees,2026,61,0.023188,0.153846,-0.019344
1,C,ridge,2026,61,0.024940,0.153062,-0.008969
0,C,mean_baseline,2026,61,0.025043,0.153032,-0.008585
4,C,mlp_neural_network,2026,61,0.028166,0.152764,-0.005055
...,...,...,...,...,...,...,...
119,WR,mlp_neural_network,2026,752,4.458717,6.176785,0.372703
118,WR,hist_gradient_boosting,2026,752,4.498560,6.191514,0.369707
117,WR,extra_trees,2026,752,4.548584,6.290809,0.349329
116,WR,ridge,2026,752,4.576843,6.287088,0.350099


## 7. Residual-based prediction intervals and model selection

Residual quantiles provide a practical approximate interval, not guaranteed calibrated coverage. Proper coverage should be measured on an independent calibration period or via conformal prediction with a suitable temporal design. Select a model per position using validation metrics, then preserve the test set for final reporting.

In [8]:
residuals=pd.concat(prediction_frames,ignore_index=True) if prediction_frames else pd.DataFrame()
interval_summary=[]
if not residuals.empty:
    for pos,g in residuals.groupby("position"):
        qlo=float(g["residual"].quantile(PREDICTION_INTERVAL_ALPHA/2))
        qhi=float(g["residual"].quantile(1-PREDICTION_INTERVAL_ALPHA/2))
        interval_summary.append({"position":pos,"residual_q_low":qlo,"residual_q_high":qhi,"n_calibration_like":len(g)})
intervals=pd.DataFrame(interval_summary)
display(intervals)
intervals.to_csv(OUT/"residual_interval_summary.csv",index=False)
if not metrics.empty:
    best=metrics.sort_values("MAE").groupby("position",as_index=False).first()
    display(best)
    best.to_csv(OUT/"best_model_by_position.csv",index=False)

,position,residual_q_low,residual_q_high,n_calibration_like
0,C,-0.009198,0.000687,61
1,CB,-0.032363,0.000000,585
2,DB,-0.025108,0.000000,92
3,DE,-0.010075,-0.000008,398
4,DL,0.000000,0.000000,15
5,DT,-0.003505,0.000000,497
6,FB,-3.249165,1.115751,22
7,FS,-0.000650,0.000000,30
8,G,-0.030238,0.000000,101
9,ILB,-0.000011,-0.000011,14


,position,model,test_season,n_test,MAE,RMSE,R2
0,C,hist_gradient_boosting,2026,61,2.244957e-02,1.540814e-01,-0.022460
1,CB,hist_gradient_boosting,2026,585,2.193242e-02,1.287878e-01,-0.213989
2,DB,extra_trees,2026,92,1.145094e-02,1.760644e-02,0.000000
3,DE,extra_trees,2026,398,2.243687e-03,7.443773e-03,0.000000
4,DL,hist_gradient_boosting,2026,15,0.000000e+00,0.000000e+00,1.000000
5,DT,ridge,2026,497,4.197227e-03,1.480192e-02,-0.212341
6,FB,mlp_neural_network,2026,22,2.074569e+00,2.553998e+00,0.072460
7,FS,extra_trees,2026,30,1.081417e-03,5.148138e-03,0.000000
8,G,mlp_neural_network,2026,101,3.168925e-02,2.294226e-01,-0.014985
9,ILB,hist_gradient_boosting,2026,14,1.104406e-08,1.104406e-08,0.000000


## 8. Save model artifacts and a model card

Saved models include preprocessing. A separate JSON card records the target, feature set, training seasons, held-out season, and the limitations that should accompany forecasts.

In [9]:
for (pos,name),bundle in fitted.items():
    safe=re.sub(r"[^A-Za-z0-9_-]+","_",f"{pos}_{name}_{TARGET}")
    joblib.dump(bundle,MODEL_DIR/f"{safe}.joblib")

model_card={
    "created_at_utc":datetime.now(timezone.utc).isoformat(),
    "data_root":str(DATA_ROOT.resolve()),
    "target":TARGET,
    "position_specific":True,
    "feature_candidates":FEATURE_CANDIDATES,
    "models_saved":[{"position":p,"model":m,"target":b["target"],"train_seasons":b["train_seasons"],"test_season":b["test_season"],"features":b["features"]} for (p,m),b in fitted.items()],
    "warnings":["Chronological holdout is a minimum viable backtest; add rolling-origin tests before production.",
                "Residual intervals are approximate and may be miscalibrated.",
                "Position labels and player-game uniqueness must be verified.",
                "Feature availability at prediction time must be audited."]
}
with open(OUT/"model_card.json","w",encoding="utf-8") as f: json.dump(model_card,f,indent=2,default=str)
print("Saved",len(fitted),"model pipelines to",MODEL_DIR)

Saved 96 model pipelines to nfl_forecasts\models


## 9. Walk-forward backtesting helper

The previous section evaluates the latest season once. This helper supports rolling-origin backtests across multiple seasons, so you can assess whether the model works across different eras and roster/coaching environments. It uses Extra Trees as a representative model to limit runtime; compare alternatives separately.

In [10]:
def walk_forward_backtest(df, features, target, min_train_seasons=3, estimator=None):
    if df.empty or "season" not in df: return pd.DataFrame()
    seasons=sorted(pd.to_numeric(df["season"],errors="coerce").dropna().unique())
    rows=[]
    if estimator is None:
        estimator=ExtraTreesRegressor(n_estimators=200,min_samples_leaf=3,random_state=SEED,n_jobs=-1)
    for test_season in seasons[min_train_seasons:]:
        train=df[pd.to_numeric(df.season,errors="coerce")<test_season].copy()
        test=df[pd.to_numeric(df.season,errors="coerce")==test_season].copy()
        train=train[train[target].notna()]
        test=test[test[target].notna()]
        if len(train)<100 or len(test)<20: continue
        pipe=Pipeline([("prep",make_preprocessor(train,features)),("model",clone(estimator))])
        try:
            pipe.fit(train[features],train[target].astype(float))
            pred=pipe.predict(test[features])
            rows.append({"test_season":int(test_season),"train_rows":len(train),"test_rows":len(test),
                         "MAE":mean_absolute_error(test[target],pred),
                         "RMSE":np.sqrt(mean_squared_error(test[target],pred))})
        except Exception as e:
            print("Walk-forward error",test_season,e)
    return pd.DataFrame(rows)

walk_forward_metrics=pd.DataFrame()
if not model_data.empty and TARGET and "season" in model_data:
    wf_source=model_data.copy()
    if "position" in wf_source:
        # Overall walk-forward for comparison; position-specific tests are done in the main loop.
        wf_source=wf_source[wf_source["position"].notna()]
    wf_features=[c for c in FEATURE_CANDIDATES if c in wf_source and c not in {"player_id","player_name","team","position","season","week","game_id","gameday",TARGET}]
    if wf_features:
        walk_forward_metrics=walk_forward_backtest(wf_source,wf_features,TARGET)
display(walk_forward_metrics)
walk_forward_metrics.to_csv(OUT/"walk_forward_metrics.csv",index=False)

,test_season,train_rows,test_rows,MAE,RMSE
0,2002,50169,17460,1.563178,3.692518
1,2003,67629,17211,1.491280,3.533194
2,2004,84840,17251,1.522063,3.634526
3,2005,102091,17334,1.475212,3.447080
4,2006,119425,17193,1.458000,3.470400
5,2007,136618,17244,1.513447,3.605459
6,2008,153862,17191,1.491461,3.497099
7,2009,171053,17669,1.560189,3.631705
8,2010,188722,17569,1.570355,3.663892
9,2011,206291,17419,1.548595,3.592488


## 10. Next-game forecast builder

A production next-game forecast needs the upcoming schedule, expected player pool, and pregame role/injury information. This section creates a forecast template from the latest row for each player and applies saved position-specific Extra Trees models where compatible. It does not invent upcoming participants or future injuries.

In [11]:
def latest_player_rows(df):
    if df.empty: return df.copy()
    idcol="player_id" if "player_id" in df else "player_name"
    sort=[c for c in [idcol,"season","week","gameday"] if c in df]
    return df.sort_values(sort).groupby(idcol,dropna=False).tail(1).copy()

next_game_template=pd.DataFrame()
if not model_data.empty:
    next_game_template=latest_player_rows(model_data)
    next_game_template["forecast_note"]="Template from latest observed player row; attach upcoming schedule and pregame availability before operational use"
    next_game_template.to_csv(OUT/"next_game_forecast_template.csv",index=False)
display(next_game_template.head())

,player_id,player_name,player_display_name,position,position_group,headshot_url,season,week,season_type,game_id,team,opponent_team,completions,attempts,passing_yards,passing_tds,passing_interceptions,sacks_suffered,sack_yards_lost,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_cpoe,passing_2pt_conversions,pacr,passing_10,passing_16,passing_20,passing_40,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,rushing_10,rushing_12,rushing_20,rushing_40,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,receiving_10,receiving_16,receiving_20,receiving_40,racr,...,fg_made,fg_att,fg_missed,fg_blocked,fg_long,fg_pct,fg_made_0_19,fg_made_20_29,fg_made_30_39,fg_made_40_49,fg_made_50_59,fg_made_60_,fg_missed_0_19,fg_missed_20_29,fg_missed_30_39,fg_missed_40_49,fg_missed_50_59,fg_missed_60_,fg_made_list,fg_missed_list,fg_blocked_list,fg_made_distance,fg_missed_distance,fg_blocked_distance,pat_made,pat_att,pat_missed,pat_blocked,pat_pct,gwfg_made,gwfg_att,gwfg_missed,gwfg_blocked,gwfg_distance,pt_att,pt_blocked,pt_long,pt_yards,pt_inside_20,pt_out_of_bounds,pt_downed,pt_touchback,pt_fair_caught,pt_returned,pt_return_yards,pt_return_tds,pt_net_yards,fantasy_points,fantasy_points_ppr,opponent,source_table,prior_mean_3,prior_std_3,prior_mean_5,prior_std_5,prior_mean_8,prior_std_8,prior_mean_career,prior_games,forecast_note
41,0,None,None,None,None,None,2000,21,POST,2000_21_BAL_NYG,NYG,BAL,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,NaN,...,0,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0.0,0.0,BAL,player_stats,0.000000,0.000000,0.00,0.000000,0.0000,0.000000,0.000000,41.0,Template from latest observed player row; atta...
55,00-0000003,A.al-Jabbar,Abdul-Karim al-Jabbar,RB,RB,https://static.www.nfl.com/image/private/f_aut...,2000,1,REG,2000_01_IND_KC,IND,KC,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,1,-2,0,0,0,0,-0.549944,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,NaN,...,0,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,-0.2,-0.2,KC,player_stats,8.600000,3.195309,7.28,2.935473,6.3125,2.856290,6.300000,13.0,Template from latest observed player row; atta...
127,00-0000007,R.Abdullah,Rabih Abdullah,RB,RB,https://static.www.nfl.com/image/private/f_aut...,2004,21,POST,2004_21_NE_PHI,NE,PHI,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,NaN,...,0,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0.0,0.0,PHI,player_stats,0.333333,0.288675,0.10,0.418330,1.1500,2.232871,0.700000,71.0,Template from latest observed player row; atta...
154,00-0000008,R.Abdullah,Rahim Abdullah,OLB,LB,https://static.www.nfl.com/image/private/f_aut...,2000,16,REG,2000_16_TEN_CLE,CLE,TEN,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,NaN,...,0,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0.0,0.0,TEN,player_stats,0.000000,0.000000,0.00,0.000000,0.0000,0.000000,0.000000,26.0,Template from latest observed player row; atta...
246,00-0000017,D.Abraham,Donnie Abraham,CB,DB,https://static.www.nfl.com/image/private/f_aut...,2004,19,POST,2004_19_NYJ_PIT,NYJ,PIT,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,NaN,...,0,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0.0,0.0,PIT,player_stats,0.000000,0.000000,0.00,0.000000,0.0000,0.000000,0.065934,91.0,Template from lates

## 11. Rest-of-season simulation

For each player, aggregate the expected outcome over remaining games. The correct production workflow predicts each scheduled future game, then sums simulated game outcomes. Until a schedule-aware feature builder is connected, this notebook provides a Monte Carlo utility that can be applied to a table with one row per player-game prediction and residual interval estimates.

In [12]:
def simulate_rest_of_season(game_predictions, prediction_col="prediction", low_col="lower", high_col="upper",
                            player_col="player_id", n_sims=10000, seed=SEED):
    """Input: one row per future player-game. Returns player-level total quantiles."""
    required=[prediction_col,player_col]
    missing=[c for c in required if c not in game_predictions]
    if missing: raise ValueError(f"Missing columns: {missing}")
    rng=np.random.default_rng(seed)
    rows=[]
    for player,g in game_predictions.groupby(player_col,dropna=False):
        mu=pd.to_numeric(g[prediction_col],errors="coerce").fillna(0).to_numpy()
        if low_col in g and high_col in g:
            lo=pd.to_numeric(g[low_col],errors="coerce").to_numpy()
            hi=pd.to_numeric(g[high_col],errors="coerce").to_numpy()
            sigma=np.maximum((hi-lo)/(2*1.2816),0)
        else:
            sigma=np.zeros(len(mu))
        draws=rng.normal(mu[None,:],sigma[None,:],size=(n_sims,len(mu)))
        totals=draws.sum(axis=1)
        rows.append({player_col:player,"remaining_games":len(g),"expected_total":float(np.mean(totals)),
                     "p10":float(np.quantile(totals,.10)),"p50":float(np.quantile(totals,.50)),
                     "p90":float(np.quantile(totals,.90))})
    return pd.DataFrame(rows)

# Example use after producing per-game predictions:
# remaining_totals = simulate_rest_of_season(future_player_game_predictions)

## 12. Next-season projection framework

Next-season modeling is a distinct problem, not a continuation of next-game prediction. A sound target is next-season total production or per-game rate, with predictors available before that season: age, prior-year and career performance, role/opportunity, draft capital, contract/roster status, team change, coaching change, and expected depth-chart competition. Rookie models should be separate and may use college/draft features.

The helper below creates lagged season-level targets, then allows a chronological model to be fit. Confirm the aggregation and target meaning before using it.

In [13]:
def make_next_season_table(player_game_df,target):
    if player_game_df.empty or target not in player_game_df or "season" not in player_game_df: return pd.DataFrame()
    idcol="player_id" if "player_id" in player_game_df else "player_name"
    keys=[idcol,"season"] + (["position"] if "position" in player_game_df else [])
    season_totals=player_game_df.groupby(keys,dropna=False)[target].sum(min_count=1).reset_index(name="season_total")
    season_totals=season_totals.sort_values([idcol,"season"])
    season_totals["next_season_target"]=season_totals.groupby(idcol,dropna=False)["season_total"].shift(-1)
    season_totals["next_season"]=season_totals.groupby(idcol,dropna=False)["season"].shift(-1)
    season_totals["prior_season_total"]=season_totals["season_total"]
    season_totals["career_mean_through_season"]=season_totals.groupby(idcol,dropna=False)["season_total"].transform(
        lambda s:s.expanding(min_periods=1).mean())
    return season_totals[season_totals["next_season"]==season_totals["season"]+1].copy()

next_season_table=make_next_season_table(player_games,TARGET) if TARGET and not player_games.empty else pd.DataFrame()
display(next_season_table.head())
next_season_table.to_csv(OUT/"next_season_training_table.csv",index=False)

,player_id,season,position,season_total,next_season_target,next_season,prior_season_total,career_mean_through_season
0,0,1999,NaN,0.0,0.0,2000.0,0.0,0.000000
2,00-0000003,1999,RB,81.9,-0.2,2000.0,81.9,81.900000
4,00-0000007,1999,RB,4.3,10.4,2000.0,4.3,4.300000
5,00-0000007,2000,RB,10.4,8.6,2001.0,10.4,7.350000
6,00-0000007,2001,RB,8.6,0.0,2002.0,8.6,7.766667


## 13. Position trend reporting

Descriptive trends are useful before predictive modeling. This exports sample sizes, means, medians, and spread by position/season for the target. Trends are not causal and can be distorted by playing time and player availability, so add opportunity rates (per snap, target, carry, or route) when those fields are reliable.

In [14]:
position_trends=pd.DataFrame()
if not player_games.empty and TARGET and "position" in player_games and "season" in player_games:
    position_trends=player_games.groupby(["position","season"],dropna=False)[TARGET].agg(
        rows="count",mean="mean",median="median",std="std",p10=lambda s:s.quantile(.10),p90=lambda s:s.quantile(.90)
    ).reset_index()
    display(position_trends.head(40))
    position_trends.to_csv(OUT/"position_target_trends.csv",index=False)
    for pos,g in position_trends.groupby("position"):
        plt.figure()
        plt.plot(g["season"],g["mean"],marker="o")
        plt.title(f"{pos}: mean {TARGET} by season")
        plt.xlabel("Season"); plt.ylabel(f"Mean {TARGET}")
        plt.tight_layout()
        plt.savefig(OUT/f"trend_{re.sub(r'[^A-Za-z0-9_-]+','_',str(pos))}_{TARGET}.png",dpi=140)
        plt.close()

,position,season,rows,mean,median,std,p10,p90
0,C,1999,187,0.000000,0.0,0.000000,0.0,0.0
1,C,2000,145,0.013103,0.0,0.157786,0.0,0.0
2,C,2001,140,0.052143,0.0,0.608509,0.0,0.0
3,C,2002,176,0.009659,0.0,0.128142,0.0,0.0
4,C,2003,232,0.000862,0.0,0.013131,0.0,0.0
5,C,2004,171,0.000000,0.0,0.000000,0.0,0.0
6,C,2005,193,0.004663,0.0,0.064783,0.0,0.0
7,C,2006,178,0.000000,0.0,0.000000,0.0,0.0
8,C,2007,182,0.003846,0.0,0.051887,0.0,0.0
9,C,2008,187,-0.001604,0.0,0.021938,0.0,0.0


## 14. Feature timing checklist and final exports

Before production use, document when each feature becomes available. The model card and metrics are saved so you can compare future runs.

In [15]:
feature_timing_checklist=pd.DataFrame([
    ["Player historical stats","After prior game final","Use lagged features only"],
    ["Opponent defense","After prior games final","Use prior-game aggregates"],
    ["Coach tenure/change","Publicly announced before kickoff","Join with valid-from date"],
    ["Injuries","Pregame report timestamp","Do not use final postgame status"],
    ["Weather","Forecast timestamp before kickoff","Do not use observed postgame weather"],
    ["Depth chart","Timestamped pregame depth chart","Avoid future updates"],
    ["Team tendency","Calculated only from prior games","Do not join full-season aggregate into earlier weeks"],
    ["Next-season roster","Only facts known at forecast cutoff","Separate retrospective analysis from real-time forecast"],
],columns=["feature_family","availability_rule","guardrail"])
display(feature_timing_checklist)
feature_timing_checklist.to_csv(OUT/"feature_timing_checklist.csv",index=False)

run_summary={
    "created_at_utc":datetime.now(timezone.utc).isoformat(),
    "data_root":str(DATA_ROOT.resolve()),
    "target":TARGET,
    "player_game_rows":len(player_games),
    "model_rows":len(model_data),
    "metrics_rows":len(metrics),
    "saved_models":len(fitted),
    "tables_loaded":{k:{"rows":len(v),"columns":len(v.columns)} for k,v in D.items()},
    "output_directory":str(OUT.resolve())
}
with open(OUT/"run_summary.json","w",encoding="utf-8") as f: json.dump(run_summary,f,indent=2,default=str)
print(json.dumps(run_summary,indent=2,default=str))

,feature_family,availability_rule,guardrail
0,Player historical stats,After prior game final,Use lagged features only
1,Opponent defense,After prior games final,Use prior-game aggregates
2,Coach tenure/change,Publicly announced before kickoff,Join with valid-from date
3,Injuries,Pregame report timestamp,Do not use final postgame status
4,Weather,Forecast timestamp before kickoff,Do not use observed postgame weather
5,Depth chart,Timestamped pregame depth chart,Avoid future updates
6,Team tendency,Calculated only from prior games,Do not join full-season aggregate into earlier...
7,Next-season roster,Only facts known at forecast cutoff,Separate retrospective analysis from real-time...


{
  "created_at_utc": "2026-10-18T08:42:25.874846+00:00",
  "data_root": "C:\\Users\\Stephen\\Desktop\\Sports\\FantasySports\\NFL\\NFL_Complete_Pipeline_v2\\nfl_data",
  "target": "fantasy_points_ppr",
  "player_game_rows": 481734,
  "model_rows": 481734,
  "metrics_rows": 120,
  "saved_models": 96,
  "tables_loaded": {
    "pbp": {
      "rows": 1293659,
      "columns": 372
    },
    "schedules": {
      "rows": 7548,
      "columns": 46
    },
    "player_stats": {
      "rows": 481734,
      "columns": 150
    },
    "player_stats_weekly": {
      "rows": 0,
      "columns": 0
    },
    "team_stats": {
      "rows": 14691,
      "columns": 138
    },
    "players": {
      "rows": 24854,
      "columns": 39
    },
    "rosters": {
      "rows": 142728,
      "columns": 36
    },
    "weekly_rosters": {
      "rows": 921705,
      "columns": 36
    },
    "snap_counts": {
      "rows": 332075,
      "columns": 16
    },
    "ngs_passing": {
      "rows": 0,
      "columns": 0
    